#Multimodal Music Analysis

In [ ]:
!pip install pandas numpy
!pip install -q sentence-transformers
!pip install wordcloud

## 1. Συλλογή Δεδομένων

Σε αυτό το βήμα φορτώνουμε τα 5 αρχεία του dataset και τα συνδέουμε μέσω του κοινού πεδίου `id`. Ορίζουμε τα paths για κάθε αρχείο που βρίσκεται στο Google Drive.

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import tarfile
import os
from sentence_transformers import SentenceTransformer
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.manifold import TSNE

drive.mount('/content/drive')

print('Libraries loaded!')

### Φόρτωση Genres & Εύρεση Top-5

Φορτώνουμε το `id_genres.csv` και βρίσκουμε τα 5 πιο συχνά μουσικά είδη. Κρατάμε μόνο τα τραγούδια που ανήκουν σε αυτά τα genres, μειώνοντας έτσι το μέγεθος του dataset στα πιο αντιπροσωπευτικά είδη.

In [ ]:
# Ορισμός Paths
BASE_PATH   = '/content/drive/MyDrive/TEDE/'

GENRES_PATH = BASE_PATH + 'id_genres.csv'
TAGS_PATH   = BASE_PATH + 'id_tags.csv'
INFO_PATH   = BASE_PATH + 'id_information.csv'
MFCC_PATH   = BASE_PATH + 'id_mfcc_stats.tsv.bz2'
LYRICS_TAR  = BASE_PATH + 'processed_lyrics.tar.gz'

OUTPUT_CSV  = BASE_PATH + 'Colab/dataset_final.csv'

print('Paths ορίστηκαν!')
print(f'Base path: {BASE_PATH}')

### Φόρτωση MFCC Stats

Φορτώνουμε τα ηχητικά χαρακτηριστικά MFCC από το συμπιεσμένο αρχείο `.tsv.bz2`. Λόγω του μεγάλου μεγέθους χρησιμοποιούμε `chunksize=500` ώστε να φορτώνουμε και να φιλτράρουμε τμηματικά, κρατώντας μόνο τα τραγούδια που υπάρχουν στα Top-5 genres.

In [ ]:
# Φόρτωση Genres & Εύρεση Top-5

print('Φόρτωση genres...')

df_genres = pd.read_csv(GENRES_PATH, sep='\t', names=['id', 'genres'], header=0)

# Cleaning
df_genres = df_genres.dropna(subset=['id', 'genres'])
df_genres = df_genres[df_genres['id'].str.strip() != '']
df_genres = df_genres[df_genres['genres'].str.strip() != '']
df_genres = df_genres.drop_duplicates(subset='id')

# Explode για να βρούμε τα Top-5 genres
df_genres_exp = df_genres.copy()
df_genres_exp['genres'] = df_genres_exp['genres'].str.split(',')
df_genres_exp = df_genres_exp.explode('genres')
df_genres_exp['genres'] = df_genres_exp['genres'].str.strip()
df_genres_exp = df_genres_exp[df_genres_exp['genres'] != '']

top5_genres = df_genres_exp['genres'].value_counts().head(5).index.tolist()
print(f'Top-5 Genres: {top5_genres}')

# Κρατάμε το primary genre (πρώτο) για κάθε τραγούδι
df_genres_exp2 = df_genres.copy()
df_genres_exp2['genres'] = df_genres_exp2['genres'].str.split(',')
df_genres_exp2 = df_genres_exp2.explode('genres')
df_genres_exp2['genres'] = df_genres_exp2['genres'].str.strip()
df_filtered = df_genres_exp2[df_genres_exp2['genres'].isin(top5_genres)][['id', 'genres']]

df_filtered = df_filtered.rename(columns={'genres': 'genre'})
df_filtered = df_filtered.drop_duplicates(subset='id')

target_ids = set(df_filtered['id'].tolist())

print(f'Τραγούδια μετά το genre filtering: {len(df_filtered)}')

In [ ]:
# ΚΕΛΙ 4: Φόρτωση MFCC Stats (chunksize για μεγάλο αρχείο)

print('Φόρτωση MFCC Stats... (παρακαλώ περίμενε)')

mfcc_chunks = []
chunk_count = 0

for chunk in pd.read_csv(MFCC_PATH, sep='\t', compression='bz2',
                          chunksize=500, header=0):

    # Μετονομασία πρώτης στήλης σε 'id'
    chunk = chunk.rename(columns={chunk.columns[0]: 'id'})

    # Cleaning
    chunk = chunk.dropna(subset=['id'])
    chunk = chunk[chunk['id'].str.strip() != '']
    chunk = chunk.drop_duplicates(subset='id')

    # Φιλτράρισμα μόνο για τα target IDs
    filtered = chunk[chunk['id'].isin(target_ids)]
    if not filtered.empty:
        mfcc_chunks.append(filtered)

    chunk_count += 1
    if chunk_count % 50 == 0:
        print(f'  Επεξεργασία chunk {chunk_count}...')

df_mfcc = pd.concat(mfcc_chunks, ignore_index=True)
df_mfcc = df_mfcc.drop_duplicates(subset='id')

# Αφαίρεση γραμμών με NaN στα features
feature_cols = [c for c in df_mfcc.columns if c != 'id']
df_mfcc = df_mfcc.dropna(subset=feature_cols, how='all')

# Συμπίεση features σε ένα string για αποθήκευση
df_mfcc['mfcc_stats'] = df_mfcc[feature_cols].astype(str).agg(','.join, axis=1)
df_mfcc = df_mfcc[['id', 'mfcc_stats']]

print(f'Τραγούδια με MFCC data: {len(df_mfcc)}')

### Φόρτωση Lyrics

Εξάγουμε τους στίχους από το αρχείο `processed_lyrics.tar.gz`. Κάνουμε iterate τα αρχεία του archive και κρατάμε μόνο τα τραγούδια που υπάρχουν και στα genres **και** στα MFCC (τριπλή τομή). Αγνοούμε αρχεία με λιγότερες από 5 λέξεις.

In [ ]:
# ΚΕΛΙ 5: Φόρτωση Lyrics από tar.gz

print('Φόρτωση Lyrics... (παρακαλώ περίμενε)')

# Intersection μέχρι τώρα: genres ∩ mfcc
ids_with_mfcc    = set(df_mfcc['id'].tolist())
target_ids_final = target_ids & ids_with_mfcc
print(f'IDs που υπάρχουν σε genres ΚΑΙ mfcc: {len(target_ids_final)}')

lyrics_dict = {}
with tarfile.open(LYRICS_TAR, 'r:gz') as tar:
    for member in tar.getmembers():
        if not member.isfile():
            continue

        song_id = os.path.splitext(os.path.basename(member.name))[0]

        if song_id not in target_ids_final:
            continue

        f = tar.extractfile(member)
        if f is None:
            continue

        lyrics = f.read().decode('utf-8', errors='ignore').strip()

        # Αγνοούμε κενά αρχεία ή πολύ μικρά (λιγότερο από 5 λέξεις)
        if len(lyrics.split()) < 5:
            continue

        lyrics_dict[song_id] = lyrics

df_lyrics = pd.DataFrame(list(lyrics_dict.items()), columns=['id', 'lyrics'])
print(f'Τραγούδια με lyrics: {len(df_lyrics)}')

### Συνένωση & Καθαρισμός Δεδομένων

Συνενώνουμε τα τρία dataframes (genres, MFCC, lyrics) με inner join ώστε να κρατήσουμε μόνο τα τραγούδια για τα οποία έχουμε και τα τρία είδη δεδομένων. Αφαιρούμε διπλότυπα και κενές εγγραφές και αποθηκεύουμε το τελικό dataset σε CSV.

In [ ]:
# ΚΕΛΙ 6: Merge, Cleaning & Αποθήκευση

print('Συνένωση δεδομένων...')

df_final = df_filtered.merge(df_mfcc,   on='id', how='inner')
df_final = df_final.merge(df_lyrics,    on='id', how='inner')

# Αφαίρεση NaN σε κρίσιμες στήλες
df_final = df_final.dropna(subset=['id', 'genre', 'lyrics', 'mfcc_stats'])

# Αφαίρεση κενών strings
df_final = df_final[df_final['lyrics'].str.strip() != '']
df_final = df_final[df_final['mfcc_stats'].str.strip() != '']

# Αφαίρεση διπλότυπων
df_final = df_final.drop_duplicates(subset='id')

# Reset index
df_final = df_final.reset_index(drop=True)

# Αποτελέσματα
print(f'\nΤελικό dataset: {len(df_final)} τραγούδια')
print(f'Στήλες: {list(df_final.columns)}')
print(f'\nΚατανομή ανά genre:')
print(df_final['genre'].value_counts())
print(f'\nΔείγμα δεδομένων:')
print(df_final[["id", "genre", "lyrics"]].head(3).to_string())

# Αποθήκευση
df_final.to_csv(OUTPUT_CSV, index=False)
print(f'\nΑποθηκεύτηκε στο: {OUTPUT_CSV}')

## 2. Εξαγωγή Χαρακτηριστικών & Embeddings

### Text Embeddings (BERT)
Χρησιμοποιούμε το προ-εκπαιδευμένο μοντέλο `all-MiniLM-L6-v2` της βιβλιοθήκης Sentence-Transformers για να δημιουργήσουμε ένα πυκνό διάνυσμα 384 διαστάσεων για κάθε τραγούδι βάσει των στίχων του. Τα lyrics περικόπτονται στους πρώτους 1000 χαρακτήρες για αποφυγή υπέρβασης του token limit.

### Audio Embeddings (Autoencoder)
Για τα ηχητικά χαρακτηριστικά εκπαιδεύουμε ένα Dense Autoencoder με Keras. Το δίκτυο συμπιέζει τα υψηλής διάστασης MFCC features σε ένα bottleneck layer 32 διαστάσεων. Χρησιμοποιούμε τον Encoder για να εξάγουμε τα τελικά Audio Embeddings.

In [ ]:
# ΚΕΛΙ 7: Text Embeddings και Audio Embeddings

print('Δημιουργία BERT embeddings για lyrics...')

# Φόρτωση μοντέλου
model = SentenceTransformer('all-MiniLM-L6-v2')

#Truncate lyrics (για αποφυγή token limits)
df_final['lyrics_short'] = df_final['lyrics'].str[:1000]

# Μετατροπή σε λίστα
lyrics_list = df_final['lyrics_short'].tolist()

# Δημιουργία embeddings
text_embeddings_bert = model.encode(
    lyrics_list,
    batch_size=32,
    show_progress_bar=True
)

# Αποθήκευση στο dataframe
df_final['text_embedding_bert'] = list(text_embeddings_bert)

# Έλεγχος
print(f'Embedding dimension: {len(df_final["text_embedding_bert"][0])}')
print(f'Συνολικά τραγούδια: {len(df_final)}')




print('Δημιουργία Audio Embeddings με Autoencoder...')

# Μετατροπή MFCC από string σε numpy array
df_final['mfcc_array'] = df_final['mfcc_stats'].apply(
    lambda x: np.array([float(v) for v in x.split(',')])
)

# Στοίχιση όλων των τραγουδιών σε ένα 2D array
X_mfcc = np.vstack(df_final['mfcc_array'].values)
print(f'MFCC matrix shape: {X_mfcc.shape}')

# Κανονικοποίηση με mean=0 και std=1
X_mfcc_mean = X_mfcc.mean(axis=0)
X_mfcc_std  = X_mfcc.std(axis=0)
X_mfcc_norm = (X_mfcc - X_mfcc_mean) / (X_mfcc_std + 1e-9)

# Ορισμός Autoencoder
input_dim = X_mfcc_norm.shape[1]
encoding_dim = 32

# Encoder
input_layer = Input(shape=(input_dim,))
x = Dense(128, activation='relu')(input_layer)
x = Dense(64, activation='relu')(x)
bottleneck = Dense(encoding_dim, activation='relu')(x)

# Decoder
x = Dense(64, activation='relu')(bottleneck)
x = Dense(128, activation='relu')(x)
output_layer = Dense(input_dim, activation='linear')(x)

# Autoencoder μοντέλο
autoencoder = Model(input_layer, output_layer)
encoder = Model(input_layer, bottleneck)

autoencoder.compile(optimizer='adam', loss='mse')

# 4. Εκπαίδευση Autoencoder
autoencoder.fit(
    X_mfcc_norm, X_mfcc_norm,
    epochs=50,
    batch_size=256,
    validation_split=0.1,
    shuffle=True,
    verbose=1
)

# Δημιουργία Audio Embeddings
audio_embeddings_ae = encoder.predict(X_mfcc_norm)
print(f'Audio Embeddings shape: {audio_embeddings_ae.shape}')

# Αποθήκευση embeddings στο df_final
df_final['audio_embedding'] = list(audio_embeddings_ae)
print('Audio embeddings προστέθηκαν στο df_final')



## 3. Οπτικοποίηση & Εξερευνητική Ανάλυση (EDA)

### Word Clouds ανά Genre
Επιλέγουμε τα δύο πιο διαφορετικά genres (rock και pop) και δημιουργούμε Word Cloud από τα tags των τραγουδιών κάθε genre. Τα tags αποτελούν user-generated περιγραφές και δίνουν μια "ανθρώπινη" εικόνα του κάθε είδους.

In [ ]:
# ΚΕΛΙ 8: Word Clouds ανά Genre
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Φόρτωση tags
df_tags = pd.read_csv(BASE_PATH + 'id_tags.csv', sep='\t', names=['id', 'tags'], header=0)

# Merge με το dataset_final
df_wc = df_final[['id', 'genre']].merge(df_tags, on='id', how='left')

# Τα 2 πιο διαφορετικά genres
GENRE_A = 'rock'
GENRE_B = 'pop'

def get_tags_for_genre(df, genre):
    subset = df[df['genre'] == genre]['tags'].dropna()
    all_tags = []
    for tag in subset:
        tags = [t.strip() for t in str(tag).split(',')]
        all_tags.extend(tags)
    return ' '.join(all_tags)

tags_rock = get_tags_for_genre(df_wc, GENRE_A)
tags_pop = get_tags_for_genre(df_wc, GENRE_B)

# Δημιουργία Word Clouds
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, tags, genre, color in zip(
    axes,
    [tags_rock, tags_pop],
    [GENRE_A, GENRE_B],
    ['Blues', 'Reds']
):
    wc = WordCloud(
        width=700, height=400,
        background_color='white',
        colormap=color,
        max_words=80
    ).generate(tags)
    ax.imshow(wc, interpolation='bilinear')
    ax.axis('off')
    ax.set_title(f'Word Cloud - {genre.upper()}', fontsize=16, fontweight='bold')

plt.suptitle('Tags per Genre', fontsize=18)
plt.tight_layout()
plt.savefig(BASE_PATH + 'Colab/wordclouds.png', dpi=150, bbox_inches='tight')
plt.show()

### Σχολιασμός Word Clouds

Το Word Cloud του Rock κυριαρχείται από tags όπως "alternative", "classic rock", "hard rock" και
"punk", αντικατοπτρίζοντας την ποικιλομορφία των υποειδών του rock. Το Word Cloud του Pop από
την άλλη κυριαρχείται από tags όπως "female vocalist", "dance pop" και "indie pop", που δείχνουν
ότι οι χρήστες περιγράφουν την pop περισσότερο με βάση τα χαρακτηριστικά του καλλιτέχνη και
τον ρυθμό παρά με υποείδη. Τα δύο Word Clouds είναι οπτικά πολύ διαφορετικά, επιβεβαιώνοντας
ότι το rock και το pop έχουν ξεκάθαρα διαφορετική "ταυτότητα" στη γνώμη των χρηστών.

### Bar Chart - Top 10 Tags
Μετράμε τη συχνότητα εμφάνισης όλων των tags σε ολόκληρο το dataset και οπτικοποιούμε τα 10 πιο συχνά. Αυτό μας δίνει μια γενική εικόνα για το πώς οι χρήστες περιγράφουν τη μουσική στο dataset.

In [ ]:
# ΚΕΛΙ 9: Bar Chart - Top 10 Tags

from collections import Counter

# Μαζεύουμε όλα τα tags από όλο το dataset
all_tags = []
for tag in df_wc['tags'].dropna():
    tags = [t.strip() for t in str(tag).split(',')]
    all_tags.extend(tags)

# Μετράμε και παίρνουμε τα Top-10
top10 = Counter(all_tags).most_common(10)
tag_names = [t[0] for t in top10]
tag_counts = [t[1] for t in top10]

# Bar Chart
fig, ax = plt.subplots(figsize=(12, 6))
bars = ax.barh(tag_names[::-1], tag_counts[::-1], color='steelblue', edgecolor='white')

# Προσθήκη αριθμών στις μπάρες
for bar, count in zip(bars, tag_counts[::-1]):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=10)

ax.set_xlabel('Συχνότητα', fontsize=12)
ax.set_title('Top-10 πιο συχνά tags στο dataset', fontsize=14, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
plt.tight_layout()
plt.savefig(BASE_PATH + 'Colab/top10_tags.png', dpi=150, bbox_inches='tight')
plt.show()

### Σχολιασμός Top-10 Tags

Τα πιο συχνά tags είναι "rock" (25.007) και "pop" (21.687), γεγονός αναμενόμενο καθώς αυτά
είναι τα κυρίαρχα genres του dataset. Τα tags "indie" (11.259) και "alternative" (10.427)
εμφανίζονται επίσης πολύ συχνά, δείχνοντας ότι μεγάλο μέρος του dataset αποτελείται από
indie/alternative μουσική που συχνά εκτείνεται και στα δύο genres. Το "female vocalists" (8.027)
στην 6η θέση είναι ενδιαφέρον καθώς είναι χαρακτηριστικό καλλιτέχνη και όχι genre,
υποδηλώνοντας ότι οι χρήστες ετικετάρουν τη μουσική με βάση και τον ερμηνευτή.

### Μείωση Διαστάσεων (t-SNE)
Εφαρμόζουμε t-SNE για να μειώσουμε τα Text και Audio Embeddings σε 2 διαστάσεις. Για λόγους ταχύτητας χρησιμοποιούμε τυχαίο δείγμα 1000 τραγουδιών. Τα scatter plots που ακολουθούν χρωματίζουν κάθε τραγούδι με βάση το genre του.

In [ ]:
# ΚΕΛΙ 10: Dimensionality Reduction

# Sample 1000 τραγούδια
df_sample = df_final.sample(n=1000, random_state=42).reset_index(drop=True)

# Μετατροπή embeddings από λίστα σε numpy array
text_emb = np.vstack(df_sample['text_embedding_bert'].values)
audio_emb = np.vstack(df_sample['audio_embedding'].values)
genres = df_sample['genre'].values

print(f'Text embeddings shape: {text_emb.shape}')
print(f'Audio embeddings shape: {audio_emb.shape}')


In [ ]:
# ΚΕΛΙ 11: t-SNE & Scatter Plots
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

unique_genres = list(df_sample['genre'].unique())
colors = cm.tab10(np.linspace(0, 1, len(unique_genres)))
genre_color_map = dict(zip(unique_genres, colors))

def plot_scatter(embeddings_2d, genres, title, filename):
    fig, ax = plt.subplots(figsize=(12, 8))
    for genre in unique_genres:
        mask = genres == genre
        ax.scatter(
            embeddings_2d[mask, 0],
            embeddings_2d[mask, 1],
            c=[genre_color_map[genre]],
            label=genre,
            alpha=0.5,
            s=15
        )
    ax.legend(title='Genre', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(BASE_PATH + f'Colab/{filename}', dpi=50, bbox_inches='tight')
    plt.show()
    print(f'{title} αποθηκεύτηκε...')

# t-SNE στα Text Embeddings
print('Εκτέλεση t-SNE σε text embeddings...')
tsne2 = TSNE(n_components=2, random_state=42, perplexity=30)
text_2d = tsne2.fit_transform(text_emb)
plot_scatter(text_2d, genres, 't-SNE - Text Embeddings (BERT)', 'tsne_text.png')

# t-SNE στα Audio Embeddings
print('Εκτέλεση t_SNE σε audio embeddings...')
tsne2 = TSNE(n_components=2, random_state=42, perplexity=30)
audio_2d = tsne2.fit_transform(audio_emb)
plot_scatter(audio_2d, genres, 't-SNE - Audio Embeddings (Autoencoder)', 'tsne_audio.png')



## **Σύγκριση Audio vs Text Embeddings στο t-SNE**

Παρατηρώντας τα δυο scatter plots, και οι δυο αναπαραστάσεις εμφανίζουν σημαντική ανάμειξη των genres χωρίς ξεκάθαρους διαχωρισμούς. Ωστόσο τα audio embeddings φαίνεται να διαχωρίζουν ελεγρώς καλύτερα τις κλάσεις σε σχέση με τα text embeddings, καθώς παρατηρούνται μικρές συγκεντρώσεις ομοειδών genres σε ορισμένες περιοχές του χώρου. Το γεγονός ότι κανένα απο τα δύο δεν σχηματίζει ξεκάθαρα clusters συμβαίνει καθώς τα genres που επιλέχθηκαν έχουν πολλα κοινά χαρακτηριστικά μεταξύ τους.

### Similarity Analysis
Για ένα τυχαίο τραγούδι βρίσκουμε τα 5 πιο όμοια τραγούδια βάσει Cosine Similarity, πρώτα χρησιμοποιώντας τα Text Embeddings και μετά τα Audio Embeddings. Συγκρίνουμε τα αποτελέσματα για να δούμε ποια modality ομαδοποιεί καλύτερα τα τραγούδια.

In [ ]:
# ΚΕΛΙ 12: Similarity Analysis

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Μετατροπή σε numpy arrays
text_emb = np.array(text_embeddings_bert)
audio_emb = np.array(audio_embeddings_ae)

# Βρίσκουμε παρόμοια τραγούδια
def find_similar(song_index, embeddings, df, top_k=5):
    # Υπολογισμός cosine similarity
    sims = cosine_similarity([embeddings[song_index]], embeddings)[0]

    similar_indices = np.argsort(sims)[::-1]
    similar_indices = similar_indices[1:top_k+1]

    return similar_indices, sims[similar_indices]

# Δοκιμή για ένα τραγούδι
song_idx = 1821
print('Αρχικό τραγούδι:')
print(df_final.iloc[song_idx][['id', 'genre']])
print('\n--- Text Similarity ---')

text_idx, text_scores = find_similar(song_idx, text_emb, df_final)

for i, score in zip(text_idx, text_scores):
    print(f'ID: {df_final.iloc[i]['id']}')
    print(f'Genre: {df_final.iloc[i]['genre']}')
    print(f'Similarity: {score:.4f}')

print('\n--- Audio Similarity ---')

audio_idx, audio_score = find_similar(song_idx, audio_emb, df_final)

for i, score in zip(audio_idx, audio_score):
    print(f"ID: {df_final.iloc[i]['id']}")
    print(f"Genre: {df_final.iloc[i]['genre']}")
    print(f"Similarity: {score:.4f}")


## **Σχολιασμός Similariry Analysis**

Η Text Similarity βρίσκει τραγούδια από διάφορα genres με scores περίπου 0.71-0.74. Παρά το γεγονός ότι το αρχικό τραγούδι είναι indie rock, κανένα από τα 5 πιο όμοια δεν ανήκει στο ίδιο genre. Αυτό δείχνει ότι το θεματικό περιεχόμενο των στίχων είναι κοινό across genres και δεν αποτελεί αξιόπιστο κριτήριο διαχωρισμού.

Η Audio Similarity βρίσκει τραγούδια rock και pop με πολύ υψηλά scores (0.97+). Αν και δεν βρίσκει indie rock τραγούδια, τα genres που επιστρέφει (rock, pop) είναι ηχητικά πολύ κοντά στο indie rock.

Τα audio embeddings δίνουν ηχητικά πιο συνεκτικά αποτελέσματα, ενώ τα text embeddings επηρεάζονται περισσότερο από το θεματικό περιεχόμενο των στίχων παρά από το μουσικό genre.